# Data summary: sample sizes, demographics and head motion

In [ ]:
import warnings
warnings.simplefilter("ignore", UserWarning)

import numpy as np
import pandas as pd

from src.data import DataLoader

## Sample demographics

In [ ]:
# ── Sample demographics summary for methods paragraph ─────────────────────────
# use_for_total=False keeps a subsample out of the combined counts, and sessions counts only subjects with a
# registry scan at those sessions (Penn_LEAD: session 1, the one every figure analyses).
REGISTRY_CONFIGS = [
    dict(label='HCP-YA',      dataset='HCP-YA', registry='scan_registry_HCP-YA.csv',       use_for_total=True),
    dict(label='HCP-YA (7T)', dataset='HCP-YA', registry='scan_registry_HCP-YA_multi.csv', use_for_total=False),
    dict(label='WAND',        dataset='WAND',    registry='scan_registry_WAND.csv',          use_for_total=True),
    dict(label='HCP-D',       dataset='HCP-D',   registry='scan_registry_HCP-D.csv',         use_for_total=True),
    dict(label='Penn_LEAD',   dataset='Penn_LEAD', registry='scan_registry_Penn_LEAD.csv',   use_for_total=True,
         sessions=('1',)),
]

rows = []
loaders = {}
for cfg in REGISTRY_CONFIGS:
    dataset = cfg['dataset']
    if dataset not in loaders:
        loaders[dataset] = DataLoader(dataset=dataset, verbose=False)
    loader = loaders[dataset]

    # Included subject IDs from scan registry CSV
    reg = pd.read_csv(str(loader.output_dir / cfg['registry']), index_col=0, dtype=str)
    if cfg.get('sessions') is not None:
        reg = reg[reg['ses'].isin(cfg['sessions'])]
    included = set(reg['sub'].unique())
    n = len(included)

    # Match to demographics
    if loader.df_behavior is not None:
        df_beh = loader.df_behavior.copy()
        df_beh.index = df_beh.index.astype(str)
        df_m = df_beh.loc[df_beh.index.intersection(included)]
    else:
        # No behaviour table: fall back to participants.tsv in the dataset base dir
        try:
            df_m = pd.read_csv(loader.base_dir / 'participants.tsv', sep='\t')
            df_m['subject_id'] = df_m['participant_id'].str.replace('sub-', '', regex=False)
            df_m = df_m.set_index('subject_id').loc[lambda d: d.index.isin(included)]
            sex_col = next((c for c in df_m.columns if c.lower() in ('sex', 'gender')), None)
            if sex_col:
                df_m = df_m.rename(columns={sex_col: 'sex'})
                df_m['sex'] = df_m['sex'].map(
                    lambda v: 0 if str(v).lower() in ('f', 'female') else
                              1 if str(v).lower() in ('m', 'male') else np.nan)
        except Exception as e:
            print(f'  {dataset}: participants.tsv not loaded ({e})')
            df_m = pd.DataFrame()

    # Age
    if 'age' in df_m.columns:
        ages      = df_m['age'].dropna()
        age_min   = ages.min()
        age_max   = ages.max()
        age_range = f"{age_min:.1f}-{age_max:.1f}"
        age_ms    = f"{ages.mean():.1f} +/- {ages.std():.1f}"
    else:
        age_min = age_max = np.nan
        age_range = age_ms = 'N/A'

    # Sex: 0=Female, 1=Male across all
    if 'sex' in df_m.columns:
        n_female   = int((df_m['sex'] == 0).sum())
        pct_female = 100 * n_female / len(df_m)
    else:
        n_female = pct_female = np.nan

    # Head motion per subject, labelled by metric because the XCP-D datasets and HCP-D report Power's FD while HCP-YA reports relative RMS.
    if {'meanFD_rest', 'meanFD_task'}.issubset(df_m.columns):
        fd = df_m[['meanFD_rest', 'meanFD_task']].mean(axis=1).dropna()   # XCP-D datasets
        metric = 'FD'
    elif 'meanFD' in df_m.columns:
        fd = df_m['meanFD'].dropna()                                     # HCP-D: one value/subject
        metric = 'FD'
    elif {'meanRelRMS_rest', 'meanRelRMS_task'}.issubset(df_m.columns):
        fd = df_m[['meanRelRMS_rest', 'meanRelRMS_task']].mean(axis=1).dropna()   # HCP-YA, 3T runs
        metric = 'relRMS'
    else:
        fd = pd.Series(dtype=float)
        metric = '-'

    if len(fd):
        mean_fd_str  = f"{fd.mean():.3f} +/- {fd.std():.3f}"
        range_fd_str = f"{fd.min():.3f}-{fd.max():.3f}"
    else:
        mean_fd_str = range_fd_str = 'N/A'

    rows.append(dict(
        label=cfg['label'], n=n, use_for_total=cfg['use_for_total'],
        age_min=age_min, age_max=age_max,
        age_range=age_range, age_ms=age_ms,
        n_female=n_female, pct_female=pct_female,
        metric=metric, mean_fd_str=mean_fd_str, range_fd_str=range_fd_str,
    ))

df_sum = pd.DataFrame(rows)
df_total = df_sum[df_sum['use_for_total']]

# Combined totals
n_total        = int(df_total['n'].sum())
age_min_all    = df_total['age_min'].min()
age_max_all    = df_total['age_max'].max()
n_female_all   = df_total['n_female'].sum()
pct_female_all = 100 * n_female_all / n_total

# ── Table ─────────────────────────────────────────────────────────────────────
hdr = f"{'Dataset':<14}  {'N':>5}  {'Age range':>12}  {'Age M+/-SD':>14}  {'N female':>8}  {'% female':>8}  {'Metric':>7}  {'Mean motion (M+/-SD)':>20}  {'Range':>14}"
sep = '-' * len(hdr)
print(hdr)
print(sep)
for _, r in df_sum.iterrows():
    nf  = f"{int(r['n_female'])}" if not np.isnan(r['n_female']) else 'N/A'
    pf  = f"{r['pct_female']:.1f}" if not np.isnan(r['pct_female']) else 'N/A'
    tag = '' if r['use_for_total'] else '*'
    print(f"{r['label'] + tag:<14}  {r['n']:>5}  {r['age_range']:>12}  {r['age_ms']:>14}  {nf:>8}  {pf:>8}  {r['metric']:>7}  {r['mean_fd_str']:>20}  {r['range_fd_str']:>14}")
print(sep)
print('* excluded from combined totals (subsumed by HCP-YA)')
print('  N is read from the scan registry on disk, which reflects the subject list in force when')
print('  that registry was exported -- NOT necessarily the current subject_ids_lowmotion.txt. Re-')
print('  export the registries to bring the two into agreement.')
print('  Motion is the per-subject mean of the rest and task columns (meanFD_task itself averages')
print('  a dataset\'s non-rest tasks). HCP-D carries a single value per subject.')
print('  Metric FD     = Power\'s framewise displacement, head radius 50 mm (XCP-D datasets, HCP-D)')
print('  Metric relRMS = mean relative RMS displacement, ~1.8x SMALLER than FD. HCP-YA only, and')
print('                  from its 3T runs, so the 7T row repeats those same 3T values.')
print()
print(f"N_TOTAL     = {n_total}")
print(f"AGE_MIN     = {age_min_all:.1f}")
print(f"AGE_MAX     = {age_max_all:.1f}")
print(f"N_FEMALE    = {int(n_female_all)}")
print(f"PCT_FEMALE  = {pct_female_all:.1f}")